# Part 1.B — Fine-tuning GPT2 with LoRA

In Part 1.A we trained a small GPT2 from scratch. Here we do something more realistic — we start from the **pre-trained full-size GPT2** (from HuggingFace) and fine-tune it on Penn TreeBank.

The problem: even the smallest GPT2 has ~117M parameters. We can't afford to retrain all of them.

The solution: **LoRA (Low Rank Adaptation)**. Instead of updating the original weights, we freeze them and add two tiny trainable matrices (`A` and `B`) next to each attention layer. We only train those.

## How LoRA works

A large weight matrix `W` (shape `d × d`) is expensive to update. LoRA assumes the update `ΔW` is **low-rank** — it can be approximated by two small matrices:

```
ΔW ≈ B · A     where A is (d × r) and B is (r × d),  r << d
```

The output of the adapted layer becomes:
```
output = W·x  +  (alpha/rank) · B·(A·x)
```

- `W·x` — original frozen weights, unchanged
- `B·(A·x)` — the LoRA adapter, the only thing we train
- `alpha/rank` — scaling factor
- `B` is initialized to **zeros** so at the start LoRA contributes nothing and we don't disturb the pre-trained model

**We apply LoRA to the query, key and value projections of every attention block.**

**Goal: PPL < 250 and lower than Part 1.A.**

## 1. Setup

In [ ]:
%pip install transformers==4.38.0

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from torch.utils.data import DataLoader
from functools import partial
import math, copy
import pandas as pd
from tqdm.notebook import tqdm
from typing import Optional, Tuple, Union
from transformers import AutoTokenizer, GPT2LMHeadModel
from transformers.models.gpt2.modeling_gpt2 import GPT2Attention

In [ ]:
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

## 2. Data Loading

Same Penn TreeBank dataset as Part 1.A. We use the same tokenizer so results are directly comparable.

In [ ]:
def read_file(path, eos_token='<eos>'):
    with open(path, 'r') as f:
        return [line.strip() + ' ' + eos_token for line in f]

train_raw = read_file('dataset/PennTreeBank/ptb.train.txt')
dev_raw   = read_file('dataset/PennTreeBank/ptb.valid.txt')
test_raw  = read_file('dataset/PennTreeBank/ptb.test.txt')
print(f'Train: {len(train_raw)}  Dev: {len(dev_raw)}  Test: {len(test_raw)}')

In [ ]:
# same tokenizer as Part 1.A so vocabulary and results are comparable
tokenizer = AutoTokenizer.from_pretrained('openai-community/gpt2')
tokenizer.pad_token = tokenizer.eos_token
print(f'Vocabulary size: {len(tokenizer)}')

In [ ]:
class PennTreeBank(data.Dataset):
    def __init__(self, corpus):
        self.sents = list(corpus)
    def __len__(self):
        return len(self.sents)
    def __getitem__(self, idx):
        return self.sents[idx]


def collate_fn(batch, tokenizer, device):
    tokenized = tokenizer(batch, padding=True, return_tensors='pt')
    input_ids = tokenized.input_ids[:, :-1].detach().clone().to(device)
    labels    = tokenized.input_ids[:, 1:].detach().clone().to(device)
    n_tokens  = torch.sum(input_ids != tokenizer.pad_token_id)
    return input_ids, labels, n_tokens


train_loader = DataLoader(
    PennTreeBank(train_raw), batch_size=8,
    collate_fn=partial(collate_fn, tokenizer=tokenizer, device=DEVICE), shuffle=True)
dev_loader   = DataLoader(
    PennTreeBank(dev_raw), batch_size=16,
    collate_fn=partial(collate_fn, tokenizer=tokenizer, device=DEVICE))
test_loader  = DataLoader(
    PennTreeBank(test_raw), batch_size=16,
    collate_fn=partial(collate_fn, tokenizer=tokenizer, device=DEVICE))
print('Dataloaders ready.')

## 3. Helper: Parameter Statistics

This tells us how many parameters the model has in total vs how many are actually being trained. The whole point of LoRA is that **trainable params should be a tiny fraction** of the total.

In [ ]:
def param_stats(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Total params     : {total:,}')
    print(f'Trainable params : {trainable:,}')
    print(f'Frozen params    : {total - trainable:,}')
    print(f'Trainable %      : {100 * trainable / total:.2f}%')

## 4. LoRA Implementation

### 4.1 CustomGPT2Attention

We extend HuggingFace's `GPT2Attention` and add LoRA adapters to it.

**Important detail from the lecturer:** unlike our Part 1.A where q, k, v had separate weight matrices, HuggingFace packs all three into **one matrix** called `c_attn` that outputs `3 × d_model`. It then splits the output into q, k, v.

So our LoRA adapter also needs to output `3 × d_model` to match:
- `lora_A`: `d_model → rank`  (shared across q, k, v)
- `lora_B`: `rank → 3 × d_model`  (one output for all three)

In [ ]:
class CustomGPT2Attention(GPT2Attention):
    def __init__(self, config, rank, alpha):
        super().__init__(config)

        self.rank    = rank
        self.scaling = alpha / rank  # scaling factor from the LoRA paper

        d_model = config.hidden_size

        # lora_A: compress input down to rank dimensions
        self.lora_A = nn.Linear(d_model, rank, bias=False)
        # lora_B: expand back up to 3*d_model (to match c_attn which covers q, k, v together)
        self.lora_B = nn.Linear(rank, 3 * d_model, bias=False)

        # A is initialized randomly, B is initialized to zeros
        # zero init means at the start LoRA contributes nothing — we don't disturb pre-trained weights
        nn.init.normal_(self.lora_A.weight, std=0.02)
        nn.init.zeros_(self.lora_B.weight)

    # forward copied from transformers 4.38.0 with one change: add LoRA delta before splitting q,k,v
    # https://github.com/huggingface/transformers/blob/v4.38.0/src/transformers/models/gpt2/modeling_gpt2.py
    def forward(
        self,
        hidden_states: Optional[Tuple[torch.FloatTensor]],
        layer_past: Optional[Tuple[torch.Tensor]] = None,
        attention_mask: Optional[torch.FloatTensor] = None,
        head_mask: Optional[torch.FloatTensor] = None,
        encoder_hidden_states: Optional[torch.Tensor] = None,
        encoder_attention_mask: Optional[torch.FloatTensor] = None,
        use_cache: Optional[bool] = False,
        output_attentions: Optional[bool] = False,
    ) -> Tuple[Union[torch.Tensor, Tuple[torch.Tensor]], ...]:
        if encoder_hidden_states is not None:
            if not hasattr(self, 'q_attn'):
                raise ValueError(
                    'If class is used as cross attention, the weights `q_attn` have to be defined. '
                    'Please make sure to instantiate class with `GPT2Attention(..., is_cross_attention=True)`.'
                )
            query = self.q_attn(hidden_states)
            key, value = self.c_attn(encoder_hidden_states).split(self.split_size, dim=2)
            attention_mask = encoder_attention_mask
        else:
            # --- LoRA modification ---
            # original: query, key, value = self.c_attn(hidden_states).split(self.split_size, dim=2)
            # we add the LoRA delta to the c_attn output before splitting into q, k, v
            lora_delta = self.scaling * self.lora_B(self.lora_A(hidden_states))
            query, key, value = (self.c_attn(hidden_states) + lora_delta).split(self.split_size, dim=2)

        query = self._split_heads(query, self.num_heads, self.head_dim)
        key   = self._split_heads(key,   self.num_heads, self.head_dim)
        value = self._split_heads(value, self.num_heads, self.head_dim)

        if layer_past is not None:
            past_key, past_value = layer_past
            key   = torch.cat((past_key,   key),   dim=-2)
            value = torch.cat((past_value, value), dim=-2)

        if use_cache is True:
            present = (key, value)
        else:
            present = None

        if self.reorder_and_upcast_attn:
            attn_output, attn_weights = self._upcast_and_reordered_attn(query, key, value, attention_mask, head_mask)
        else:
            attn_output, attn_weights = self._attn(query, key, value, attention_mask, head_mask)

        attn_output = self._merge_heads(attn_output, self.num_heads, self.head_dim)
        attn_output = self.c_proj(attn_output)
        attn_output = self.resid_dropout(attn_output)

        outputs = (attn_output, present)
        if output_attentions:
            outputs += (attn_weights,)

        return outputs

### 4.2 GPT2_LoRA

We extend `GPT2LMHeadModel` and swap out every attention block with our `CustomGPT2Attention`.

We **copy the pre-trained weights** from the original block using `load_state_dict(..., strict=False)`. The `strict=False` is needed because the new block has extra LoRA parameters that aren't in the original weights — that's fine, they were already initialized above.

In [ ]:
class GPT2_LoRA(GPT2LMHeadModel):
    def __init__(self, *model_args, rank, alpha, **model_kwargs):
        super().__init__(*model_args, **model_kwargs)

        for block in self.transformer.h:
            # build a new attention block with LoRA
            new_attn = CustomGPT2Attention(self.config, rank=rank, alpha=alpha)
            # copy the pre-trained weights from the original block (LoRA params are skipped)
            new_attn.load_state_dict(block.attn.state_dict(), strict=False)
            # replace the original block
            block.attn = new_attn

    def forward(self, *args, **kwargs):
        return super().forward(*args, **kwargs)

## 5. Load Pre-trained Model and Freeze Weights

We load the full pre-trained GPT2 from HuggingFace, then:
1. **Freeze everything** — set `requires_grad = False` on all parameters
2. **Unfreeze only LoRA** — re-enable `requires_grad = True` on `lora_A` and `lora_B`

This way only the tiny adapters are updated during training.

In [ ]:
def build_lora_model(rank, alpha):
    model = GPT2_LoRA.from_pretrained('openai-community/gpt2', rank=rank, alpha=alpha)
    model.to(DEVICE)

    # step 1: freeze everything
    for param in model.parameters():
        param.requires_grad = False

    # step 2: unfreeze only the LoRA adapters
    for module in model.modules():
        if hasattr(module, 'lora_A'):
            for param in module.lora_A.parameters():
                param.requires_grad = True
        if hasattr(module, 'lora_B'):
            for param in module.lora_B.parameters():
                param.requires_grad = True

    return model


# quick test: build a model and verify only lora layers are trainable
test_model = build_lora_model(rank=4, alpha=8)
print('Trainable layers:')
for name, param in test_model.named_parameters():
    if param.requires_grad:
        print(f'  {name}')
print()
param_stats(test_model)
del test_model

## 6. Training and Evaluation

These loops are slightly different from Part 1.A. The HuggingFace model **computes the loss internally** when we pass `labels`, so we don't need a separate `criterion`.

Padding tokens are replaced with `-100` so the model ignores them (HuggingFace ignores `-100` by default).

In [ ]:
def train_loop(data, optimizer, model, tokenizer):
    model.train()
    loss_array, n_tokens_list = [], []
    pbar = tqdm(data, desc='Training', leave=False)

    for i, (input_ids, _, n_tokens) in enumerate(pbar):
        optimizer.zero_grad()

        # HuggingFace manages the label shift internally — just pass input as labels
        labels = input_ids.clone().detach()
        # replace pad tokens with -100 so they're ignored in the loss
        labels[labels == tokenizer.pad_token_id] = -100

        output = model(input_ids, labels=labels)
        output.loss.backward()
        optimizer.step()

        loss_array.append(output.loss.item() * n_tokens)
        n_tokens_list.append(n_tokens)
        if i % 100 == 0:
            pbar.set_postfix(loss=(sum(loss_array) / sum(n_tokens_list)).item())

    return sum(loss_array) / sum(n_tokens_list)


def eval_loop(data, model, tokenizer):
    model.eval()
    loss_array, n_tokens_list = [], []

    with torch.no_grad():
        for input_ids, _, n_tokens in tqdm(data, desc='Evaluating', leave=False):
            labels = input_ids.clone().detach()
            labels[labels == tokenizer.pad_token_id] = -100
            output = model(input_ids, labels=labels)
            loss_array.append(output.loss.item() * n_tokens)
            n_tokens_list.append(n_tokens)

    avg_loss = sum(loss_array) / sum(n_tokens_list)
    ppl      = math.exp(avg_loss)
    return ppl, avg_loss

### Experiment runner

Same pattern as Part 1.A — builds the model, trains with early stopping, evaluates on test, logs results.

In [ ]:
def run_experiment(name, lr, rank, alpha, n_epochs=100, patience=3):
    print(f'\n{"-" * 55}')
    print(f'  {name}')
    print(f'  lr={lr}  rank={rank}  alpha={alpha}')
    print(f'{"-" * 55}')

    model     = build_lora_model(rank=rank, alpha=alpha)
    optimizer = optim.AdamW(
        (p for p in model.parameters() if p.requires_grad), lr=lr
    )
    param_stats(model)

    best_ppl, best_model, pat = math.inf, None, patience

    for epoch in (pbar := tqdm(range(n_epochs), desc='Epochs')):
        train_loop(train_loader, optimizer, model, tokenizer)
        ppl_dev, _ = eval_loop(dev_loader, model, tokenizer)
        pbar.set_description(f'Dev PPL: {ppl_dev:.1f}')

        if ppl_dev < best_ppl:
            best_ppl   = ppl_dev
            best_model = copy.deepcopy(model).to('cpu')
            pat        = patience
        else:
            pat -= 1
            if pat <= 0:
                break

    best_model.to(DEVICE)
    ppl_test, _ = eval_loop(test_loader, best_model, tokenizer)
    best_model.to('cpu')

    print(f'  Best Dev PPL : {best_ppl:.2f}')
    print(f'  Test PPL     : {ppl_test:.2f}')

    results.append({
        'Experiment': name,
        'lr'        : lr,
        'rank'      : rank,
        'alpha'     : alpha,
        'Dev PPL'   : round(best_ppl, 2),
        'Test PPL'  : round(ppl_test, 2),
    })
    return best_model, best_ppl, ppl_test

In [ ]:
# reset before running experiments
results = []

---
## Step 0 — Find a Good Learning Rate

We keep rank and alpha fixed and just search for a good `lr`. AdamW with pre-trained models usually needs a very small learning rate so we don't destroy the pre-trained weights.

In [ ]:
for lr in [1e-3, 1e-4, 5e-5]:
    run_experiment(
        name=f'Baseline  lr={lr}',
        lr=lr, rank=4, alpha=8,
    )

---
## Step 1 — Tune Rank

**Rank** controls the size of the adapters:
- Low rank (e.g. 4) → fewer parameters, less expressive
- High rank (e.g. 16) → more parameters, more expressive but closer to full fine-tuning

Use the best `lr` from Step 0. Keep `alpha = 2 × rank` (a common starting rule).

In [ ]:
# UPDATE best_lr from Step 0
best_lr = 1e-4  # UPDATE

for rank in [4, 8, 16]:
    run_experiment(
        name=f'rank={rank}',
        lr=best_lr, rank=rank, alpha=rank * 2,
    )

---
## Step 2 — Tune Alpha

**Alpha** is a scaling factor. The actual scale applied to the LoRA output is `alpha / rank`:
- Higher alpha → adapter has more influence on the output
- Lower alpha → adapter contributes less, pre-trained weights dominate

Use the best `lr` and `rank` from above.

In [ ]:
# UPDATE from Steps 0-1
best_lr   = 1e-4  # UPDATE
best_rank = 8     # UPDATE

for alpha in [8, 16, 32]:
    run_experiment(
        name=f'alpha={alpha}',
        lr=best_lr, rank=best_rank, alpha=alpha,
    )

---
## Results Summary

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
df = pd.DataFrame(results)
display(df)